In [1]:
import numpy as np
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
import pandas as pd
import json

repo_root = Path("/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder")
os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root) + os.pathsep + os.environ.get("PYTHONPATH", "")

import LEAD as lead


# Important variables
cwd = Path.cwd()
SNRs = np.array([-np.inf,-13,-11,-9,-7,-5,-3])
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [2]:
# def metastability_score(f, df, x_grid, alpha=1.0):
#     cost = f(x_grid)**2 + alpha * df(x_grid)**2   # alpha à calibrer
#     return -np.min(cost)   # plus c'est grand, plus on est près d'un ghost

def metastability_score(f, df, x_grid, tau):
    distance_to_critical = f(x_grid)**2 + df(x_grid)**2
    normalized_distance = distance_to_critical * tau**2 # between 0 (critical) and 1 (linear dynamics)
    return 1-np.min(normalized_distance) 

# Metastability Score

In [4]:
task = "Active"
period = 'late'
file_path = Path(f'Fit_{task}_{period}')

metastability_scores = {}

# load the name of the best model for each participant
json_path = cwd / f"5CV_{task}_{period}.json"
with open(json_path, "r") as f:
    data_5cv = json.load(f)
    results_cv = data_5cv.get("results_cv")
    print("Loaded results_cv:", results_cv)

for part in range(20):
    # --- Check Best Model ---
    best_model = results_cv[part]
    print(f'part {part} lets go, best model is {best_model}')

    # --- Load Data ---
    data_ref = f'myEpochs_{task}/Epoch_{SubIDs[part]}-epo.fif'
    epochs_file = cwd.parents[0] / data_ref

    # --- Load MLE Parameters for the best model ---
    if best_model == "GainModulation":
        best_model_mle = lead.model.StratifiedGainModulation(
            tau=10, process_noise=0.1, measure_noise=0.1, threshold=1, sharpness=5)
        param_path = file_path / f"GainModulatedModel_part{part}_params"
        best_model_mle.load_params(param_path)
    elif best_model == "NonLinear1":
        best_model_mle = lead.model.StratifiedNonLinear1(
            tau=10, process_noise=0.1, measure_noise=0.1, threshold=1, sharpness=5, gain=0)
        param_path = file_path / f"GainFixedModel_part{part}_params"
        best_model_mle.load_params(param_path)
    else:
        best_model_mle = lead.model.StratifiedLinear(
            tau=10, process_noise=0.1, measure_noise=0.1)
        param_path = file_path / f"LinearModel_part{part}_params"
        best_model_mle.load_params(param_path)

    x_grid = np.linspace(-0.5, 2, 1000)
    max_metascore = -np.inf
    for signal_category in range(1, 5):

        if best_model == "GainModulation":
            w1, g1 = getattr(best_model_mle, f'w{signal_category}'), getattr(best_model_mle, f'g{signal_category}')
            w2, g2 = getattr(best_model_mle, f'w{signal_category+1}'), getattr(best_model_mle, f'g{signal_category+1}')
        elif best_model == "NonLinear1":
            w1, g1 = getattr(best_model_mle, f'w{signal_category}'), best_model_mle.gain
            w2, g2 = getattr(best_model_mle, f'w{signal_category+1}'), best_model_mle.gain
        else:
            w1, g1 = getattr(best_model_mle, f'w{signal_category}'), 0
            w2, g2 = getattr(best_model_mle, f'w{signal_category+1}'), 0

        # Travel on the (w1, g1) -> (w2, g2) segment
        for t in np.linspace(0, 1, 1000):
            
            w, g = w1 + t*(w2-w1), g1 + t*(g2-g1)
            threshold = best_model_mle.threshold if hasattr(best_model_mle, 'threshold') else 1
            sharpness = best_model_mle.sharpness if hasattr(best_model_mle, 'sharpness') else 5
            interp_model = lead.model.GainModulation(
                tau=best_model_mle.tau, process_noise=best_model_mle.process_noise, measure_noise=best_model_mle.measure_noise, 
                threshold=threshold, sharpness=sharpness, 
                input_weight=w, gain=g)

            # --- Compute Metastability Score ---
            model_func = interp_model.core
            def f(states, input_value=0, signal_category=signal_category):
                return model_func(states, input_value * np.ones_like(states), signal_category) - states
            def df(states, input_value=0, signal_category=signal_category):
                epsilon = 1e-5
                return (f(states + epsilon, input_value, signal_category) - f(states - epsilon, input_value, signal_category)) / (2*epsilon)

            # signal_category is not used in model.GainModulation 
            metascore = metastability_score(lambda x: f(x, input_value=1, signal_category=signal_category), 
                                            lambda x: df(x, input_value=1, signal_category=signal_category), 
                                            x_grid, tau=best_model_mle.tau)
            max_metascore = max(max_metascore, metascore)

    metastability_scores[part] = max_metascore
    print(f"Part {part} - Metastability Score: {max_metascore}")

    # Save summary CSV
    df_results = pd.DataFrame.from_dict(
        metastability_scores, orient="index", columns=["metastability_score"]
    ).sort_index()
    df_results.to_csv(file_path / "metastability_scores_bestmodel.csv")

Loaded results_cv: ['GainModulation', 'NonLinear1', 'GainModulation', 'NonLinear1', 'NonLinear1', 'GainModulation', 'NonLinear1', 'NonLinear1', 'GainModulation', 'GainModulation', 'NonLinear1', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation', 'GainModulation']
part 0 lets go, best model is GainModulation
Part 0 - Metastability Score: 0.9996545412077527
part 1 lets go, best model is NonLinear1
Part 1 - Metastability Score: 0.8117043708664976
part 2 lets go, best model is GainModulation
Part 2 - Metastability Score: 0.9946959087610071
part 3 lets go, best model is NonLinear1
Part 3 - Metastability Score: 0.9186294823244963
part 4 lets go, best model is NonLinear1
Part 4 - Metastability Score: 0.8204567060682888
part 5 lets go, best model is GainModulation
Part 5 - Metastability Score: 0.999999833971126
part 6 lets go, best model is NonLinear1
Part 6 - Metastability Score: 0.9999954445882904
par